# Agent 4 — Inject KPI Issues Into Gold Data

Modifies specific Gold records to create controlled
KPI validation failures for Agent 4 testing.

IMPORTANT: Uses fixed dates aligned with pilot Gold data.
Run ONLY ONCE. Re-running will overwrite injections.

Scenarios:
  Scenario A (2026-06-22): return_rate > 1.0 on 3 stores
  Scenario B (2026-06-23): avg_basket_value 60% below normal
  Scenario C (2026-06-24): net_quantity_change formula wrong

In [0]:
from datetime import date, datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

GOLD_STORE = f"{CATALOG}.{SCHEMA}.uc2_gold_daily_store_metrics"
GOLD_INV   = f"{CATALOG}.{SCHEMA}.uc2_gold_daily_inventory_summary"

# Fixed dates aligned with pilot data
SCENARIO_A_DATE = date(2026, 6, 22)   # Cross-KPI impossible
SCENARIO_B_DATE = date(2026, 6, 23)   # Anomaly basket drop
SCENARIO_C_DATE = date(2026, 6, 24)   # Formula mismatch

print("=" * 60)
print("AGENT 4 — KPI ISSUE INJECTION")
print("=" * 60)
print(f"Scenario A: {SCENARIO_A_DATE}  return_rate > 1.0")
print(f"Scenario B: {SCENARIO_B_DATE}  avg_basket_value -60%")
print(f"Scenario C: {SCENARIO_C_DATE}  net_quantity_change wrong")
print("=" * 60)
print()

# Verify data exists before injecting
print("Pre-flight checks:")
for dt in [SCENARIO_A_DATE, SCENARIO_B_DATE, SCENARIO_C_DATE]:
    store_count = spark.table(GOLD_STORE).filter(
        F.col("business_date") == F.lit(str(dt))
    ).count()
    inv_count = spark.table(GOLD_INV).filter(
        F.col("business_date") == F.lit(str(dt))
    ).count()
    emoji = "✅" if store_count > 0 else "❌"
    print(
        f"  {emoji} {dt}  "
        f"store_rows={store_count}  "
        f"inv_rows={inv_count}"
    )

## Scenario A: Impossible return_rate (2026-06-22)

Sets return_count > total_transactions for 3 stores.
This makes return_rate exceed 1.0 which is mathematically impossible.
Tests Cross-KPI consistency check.

In [0]:
print(f"Injecting Scenario A on: {SCENARIO_A_DATE}")

gold_a = spark.table(GOLD_STORE).filter(
    F.col("business_date") == F.lit(str(SCENARIO_A_DATE))
)

row_count = gold_a.count()
print(f"Existing rows: {row_count}")

if row_count == 0:
    print("❌ No data found. Check that Agent 2 data generation ran.")
else:
    # Show current values before injection
    print("\nBefore injection (top 5):")
    gold_a.select(
        "store_id", "total_transactions", "return_count", "return_rate"
    ).orderBy("store_id").show(5, truncate=False)

    # Delete existing rows for this date
    spark.sql(f"""
        DELETE FROM {GOLD_STORE}
        WHERE business_date = '{SCENARIO_A_DATE}'
    """)
    print(f"Deleted {row_count} rows for {SCENARIO_A_DATE}")

    # Inject: make return_count > total_transactions for 3 stores
    window = Window.orderBy("store_id")

    injected = gold_a \
        .withColumn("row_num", F.row_number().over(window)) \
        .withColumn(
            "return_count",
            F.when(
                F.col("row_num") <= 3,
                # Set return_count to 110% of total_transactions
                (F.col("total_transactions") * 1.1).cast("long")
            ).otherwise(F.col("return_count"))
        ).withColumn(
            "return_rate",
            F.col("return_count").cast("double") /
            F.col("total_transactions").cast("double")
        ).drop("row_num")

    injected.write \
            .mode("append") \
            .partitionBy("business_date") \
            .saveAsTable(GOLD_STORE)

    print(f"Written {injected.count()} rows back with injection")

    # Verify
    print("\nAfter injection (stores with return_rate > 1.0):")
    spark.table(GOLD_STORE).filter(
        (F.col("business_date") == F.lit(str(SCENARIO_A_DATE))) &
        (F.col("return_rate") > 1.0)
    ).select(
        "store_id", "total_transactions", "return_count",
        F.round("return_rate", 4).alias("return_rate")
    ).show(truncate=False)

    print("✅ Scenario A injected.")

## Scenario B: avg_basket_value 60% Drop (2026-06-23)

Reduces avg_basket_value and total_net_amount by 60%.
Simulates a pricing error or currency conversion mistake.
Tests Anomaly detection check.

In [0]:
print(f"Injecting Scenario B on: {SCENARIO_B_DATE}")

gold_b = spark.table(GOLD_STORE).filter(
    F.col("business_date") == F.lit(str(SCENARIO_B_DATE))
)

row_count = gold_b.count()
print(f"Existing rows: {row_count}")

if row_count == 0:
    print("❌ No data found. Check that Agent 2 data generation ran.")
else:
    # Show current values before injection
    print("\nBefore injection:")
    gold_b.agg(
        F.round(F.avg("avg_basket_value"), 2).alias("avg_basket"),
        F.round(F.sum("total_net_amount"), 2).alias("total_net")
    ).show(truncate=False)

    # Delete existing rows
    spark.sql(f"""
        DELETE FROM {GOLD_STORE}
        WHERE business_date = '{SCENARIO_B_DATE}'
    """)
    print(f"Deleted {row_count} rows for {SCENARIO_B_DATE}")

    # Inject: reduce avg_basket_value by 60%
    injected = gold_b \
        .withColumn("avg_basket_value",
                    F.round(F.col("avg_basket_value") * 0.40, 2)) \
        .withColumn("total_net_amount",
                    F.round(F.col("total_net_amount") * 0.40, 2)) \
        .withColumn("total_gross_amount",
                    F.round(F.col("total_gross_amount") * 0.40, 2))

    injected.write \
            .mode("append") \
            .partitionBy("business_date") \
            .saveAsTable(GOLD_STORE)

    print(f"Written {injected.count()} rows back with injection")

    # Verify
    print("\nAfter injection:")
    spark.table(GOLD_STORE).filter(
        F.col("business_date") == F.lit(str(SCENARIO_B_DATE))
    ).agg(
        F.round(F.avg("avg_basket_value"), 2).alias("avg_basket"),
        F.round(F.sum("total_net_amount"), 2).alias("total_net")
    ).show(truncate=False)

    print("✅ Scenario B injected.")

## Scenario C: net_quantity_change Formula Wrong (2026-06-24)

Adds a fixed error to net_quantity_change so it no longer equals
total_quantity_in - total_quantity_out.
Tests Formula validation check on inventory Gold.

In [0]:
print(f"Injecting Scenario C on: {SCENARIO_C_DATE}")

gold_inv_c = spark.table(GOLD_INV).filter(
    F.col("business_date") == F.lit(str(SCENARIO_C_DATE))
)

row_count = gold_inv_c.count()
print(f"Existing inventory rows: {row_count}")

if row_count == 0:
    print("❌ No inventory data found. Check that Agent 2 data generation ran.")
else:
    # Show current values before injection
    print("\nBefore injection (check formula holds):")
    gold_inv_c.select(
        "warehouse_id",
        "event_type",
        "total_quantity_in",
        "total_quantity_out",
        "net_quantity_change",
        (F.col("total_quantity_in") - F.col("total_quantity_out"))
         .alias("expected_net")
    ).withColumn(
        "formula_error",
        F.abs(
            F.col("net_quantity_change") -
            (F.col("total_quantity_in") - F.col("total_quantity_out"))
        )
    ).orderBy("formula_error", ascending=False) \
     .show(5, truncate=False)

    # Delete existing rows
    spark.sql(f"""
        DELETE FROM {GOLD_INV}
        WHERE business_date = '{SCENARIO_C_DATE}'
    """)
    print(f"Deleted {row_count} rows for {SCENARIO_C_DATE}")

    # Inject: add error to net_quantity_change
    injected = gold_inv_c.withColumn(
        "net_quantity_change",
        F.col("net_quantity_change") + F.lit(9999)
    )

    injected.write \
            .mode("append") \
            .partitionBy("business_date") \
            .saveAsTable(GOLD_INV)

    print(f"Written {injected.count()} rows back with injection")

    # Verify
    print("\nAfter injection (formula error should be 9999):")
    spark.table(GOLD_INV).filter(
        F.col("business_date") == F.lit(str(SCENARIO_C_DATE))
    ).select(
        "warehouse_id",
        "event_type",
        "net_quantity_change",
        (F.col("total_quantity_in") - F.col("total_quantity_out"))
         .alias("expected_net"),
        F.abs(
            F.col("net_quantity_change") -
            (F.col("total_quantity_in") - F.col("total_quantity_out"))
        ).alias("formula_error")
    ).orderBy("warehouse_id", "event_type") \
     .show(10, truncate=False)

    print("✅ Scenario C injected.")

## Final Verification

In [0]:
print("=" * 65)
print("INJECTION VERIFICATION SUMMARY")
print("=" * 65)

print("\nScenario A: return_rate > 1.0 on 2026-06-22")
result_a = spark.table(GOLD_STORE).filter(
    (F.col("business_date") == F.lit(str(SCENARIO_A_DATE))) &
    (F.col("return_rate") > 1.0)
).count()
print(
    f"  Stores with impossible return_rate: {result_a}  "
    f"{'✅ Injected' if result_a > 0 else '❌ Not found'}"
)

print(f"\nScenario B: avg_basket_value drop on 2026-06-23")
result_b = spark.table(GOLD_STORE).filter(
    F.col("business_date") == F.lit(str(SCENARIO_B_DATE))
).agg(
    F.round(F.avg("avg_basket_value"), 2).alias("avg_basket")
).collect()[0]["avg_basket"]
expected_normal = 220.0   # approximate historical baseline
drop_pct = (expected_normal - result_b) / expected_normal * 100 if result_b else 0
print(
    f"  Average basket value: £{result_b}  "
    f"({drop_pct:.1f}% below normal)  "
    f"{'✅ Injected' if drop_pct > 40 else '❌ Not enough drop'}"
)

print(f"\nScenario C: formula mismatch on 2026-06-24")
result_c = spark.table(GOLD_INV).filter(
    F.col("business_date") == F.lit(str(SCENARIO_C_DATE))
).select(
    F.abs(
        F.col("net_quantity_change") -
        (F.col("total_quantity_in") - F.col("total_quantity_out"))
    ).alias("error")
).agg(
    F.avg("error").alias("avg_error")
).collect()[0]["avg_error"]
print(
    f"  Average formula error: {result_c}  "
    f"{'✅ Injected' if result_c and result_c > 100 else '❌ Not found'}"
)

print("\nGold store data for controlled dates:")
spark.sql(f"""
    SELECT
        business_date,
        COUNT(*)                              AS store_count,
        ROUND(AVG(avg_basket_value), 2)       AS avg_basket,
        ROUND(AVG(return_rate), 4)            AS avg_return_rate,
        SUM(CASE WHEN return_rate > 1.0
                 THEN 1 ELSE 0 END)           AS impossible_return_rates,
        ROUND(SUM(total_net_amount), 2)       AS total_net
    FROM {GOLD_STORE}
    WHERE business_date IN (
        '2026-06-22', '2026-06-23', '2026-06-24'
    )
    GROUP BY business_date
    ORDER BY business_date
""").show(truncate=False)

print("\n" + "=" * 65)
print("INJECTION COMPLETE")
print("=" * 65)